# Compile and evaluate

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alx87grd/minilink/blob/main/examples/tutorial/core/06_compile_and_evaluate.ipynb)

From a problem and a controller to a score: `problem.compile(law % dt).evaluate()`, its record, its gradient, Monte Carlo, and the evaluator underneath.

In [ ]:
# Local conda: minilink already installed. Colab: clone + path + meshcat.
import sys

if "google.colab" in sys.modules:
    get_ipython().run_line_magic("matplotlib", "inline")
    get_ipython().system("git clone https://github.com/alx87grd/minilink")
    sys.path.insert(0, "/content/minilink")
    get_ipython().system("pip install -q meshcat")

## 1. Compile a controller on a problem, then evaluate it

`problem.compile(controller)` closes the controller on the problem's plant and compiles the
loop once, into a `ProblemEvaluator`. Its `evaluate()` runs the loop over the horizon and
scores the run by the problem's own formula: the cost $J$, whether the run stayed in `X`
and `U`, whether it ended in `Xf`. A controller on a clock (`law % dt`) runs the sampled
loop at its own period.

The task: bring an inverted pendulum back up from a 0.4 rad tilt, judged by a quadratic
cost, and end within 0.05 of the upright after four seconds. The candidate is a digital PD
law at 100 Hz.

In [ ]:
import numpy as np

from minilink import PID, BallSet, InvertedPendulum, PlanningProblem, QuadraticCost

plant = InvertedPendulum()
Q = np.diag([10.0, 1.0])
R = 0.1 * np.eye(1)
cost = QuadraticCost.from_system(plant, Q=Q, R=R)
problem = PlanningProblem(
    plant,
    cost=cost,
    x_start=np.array([0.4, 0.0]),
    tf=4.0,
    Xf=BallSet(np.zeros(2), 0.05),
)

pd = PID(Kp=20.0, Kd=4.0)
pe = problem.compile(pd % 0.01)
print(pe.evaluate())

`pe.trajectory()` is the same run with every block output recorded, a `Trajectory` of the
loop; `pe.loop` is the loop the evaluator built, which plots it.

In [ ]:
traj = pe.trajectory()

In [ ]:
pe.loop.plot_trajectory(traj)

## 2. The `Evaluation` record and the parameter tree

`evaluate` returns an `Evaluation`: one row per run of `cost`, `J` (the cost plus a failure
charge when the problem asks for it), `feasibility_margin` and `failed`,
`reachability_margin` and `reached`, the starts `x0`; and the statistics over the rows
(`objective`, `mean`, `std`, `worst`, `cvar(alpha)`, `success_rate`, `n_trials`).

`pe.params` is every number of the run in one tree: each block of the loop by id, the
problem's leaves (`cost`, the sets, `x_start`, `t_start`, `discount_rate`), and the loop's
free inputs (here the reference `r`). `evaluate(params)` takes a partial tree and merges it
over these, so a what-if is one call.

In [ ]:
list(pe.params)

In [ ]:
report = pe.evaluate()
heavier = pe.evaluate({"plant": {"m": 1.2}})
report.J, report.reached, report.feasibility_margin, heavier.J

## 3. Gradients of the score

On JAX the whole evaluation traces, so `jax.grad` of the `objective` gives
$\partial J / \partial p$ for every leaf of the tree at once: the gains, the plant's
physical parameters, the start, the cost weights. This is the entry point of gradient-based
tuning and co-design (`07_program`).

In [ ]:
import jax

dJ = jax.grad(lambda params: pe.evaluate(params).objective)(pe.params)
dJ["controller"]["Kp"], dJ["controller"]["Kd"], dJ["plant"]["m"]

## 4. Monte Carlo on a stochastic problem

The same call scores a stochastic problem: `evaluate(n_trials=, key=)` draws the starts and
the parameter laws from one key and runs the trials as one batch. Here the start is uniform
in a box around the tilt and the mass is uncertain by $\pm 20\%$.

In [ ]:
from minilink import Uniform

random_problem = problem.replace(
    x_start=Uniform([0.2, -0.2], [0.6, 0.2]),
    params={"plant": {"m": Uniform([0.8], [1.2])}},
)
monte_carlo = random_problem.compile(pd % 0.01).evaluate(n_trials=200, key=1)
print(monte_carlo)

In [ ]:
(
    monte_carlo.mean,
    monte_carlo.std,
    monte_carlo.worst,
    monte_carlo.cvar(0.1),
    monte_carlo.success_rate,
)

## 5. Side by side: `compare`

`compare(problem, name=controller, ..., dt=)` scores several controllers on the same
problem and the same draws, and prints one table. The LQR gain designed for the same $Q$
and $R$ is the benchmark the PD and PID laws are measured against.

In [ ]:
from minilink import compare
from minilink.control import lqr

comparison = compare(
    problem,
    LQR=lqr(plant, Q, R) % 0.01,
    PD=pd % 0.01,
    PID=PID(Kp=20.0, Ki=10.0, Kd=4.0) % 0.01,
    dt=0.01,
)
print(comparison)

## 6. Under the hood: `sys.compile` and the evaluator

Every tool above sits on one lower layer. `sys.compile(backend=)` lowers any system, leaf or
diagram, into a flat evaluator with a few primitives, each taking `params=None` for the
compiled-in numbers:

| Primitive | Meaning |
| --- | --- |
| `f(x, u, t, params)` | $\dot x$ |
| `outputs(x, u, t, params)` | the boundary ports, a dict |
| `step(x, u, t, params, dt=)` | the state one period later, `u` held (RK4 by default) |
| `rollout(x0, u_seq, t_start, dt, params)` | many steps in one call, shape `(N + 1, n)` |
| `rollout_batch(x0s, u_sequences, dt=, n_steps=)` | a batch of rollouts |
| `jacobian("f", "x")` | $\partial f / \partial x$ as a function of `(x, u, t, params)` |

On JAX each primitive is jit-compiled and still composes inside `jax.grad`, `jax.vmap` and
an outer `jax.jit`.

In [ ]:
evaluator = plant.compile(backend="jax")

x = np.array([0.4, 0.0])
u = np.array([0.0])
(
    evaluator.f(x, u),
    evaluator.step(x, u, 0.0, dt=0.01),
    evaluator.jacobian("f", "x")(x, u, 0.0, None),
)

In [ ]:
u_seq = np.zeros((100, 1))
x_path = evaluator.rollout(x, u_seq, 0.0, 0.01)

x0s = np.array([[0.1, 0.0], [0.2, 0.0], [0.3, 0.0]])
x_paths = evaluator.rollout_batch(x0s, dt=0.01, n_steps=100)
x_path.shape, x_paths.shape

Next, [`07_program`](07_program.ipynb): turn the evaluation into a `MathematicalProgram` and
solve it with an `Optimizer`.